In [1]:
import os
import numpy as np
from PIL import Image
import random
import torchvision.transforms as transforms
from torchvision.transforms import functional as TF

input_dir = 'dft/original_dft_images'
output_dir = 'aug/original_dft_images'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def add_gaussian_noise(image, noise_factor=0.05):
    """Add Gaussian noise to simulate DFT perturbations."""
    image_np = np.array(image)
    noise = np.random.randn(*image_np.shape) * noise_factor * 255
    noisy_image = np.clip(image_np + noise, 0, 255).astype(np.uint8)
    return TF.to_pil_image(noisy_image)

def add_frequency_shift(image, max_shift=10):
    """Simulate slight shifts in frequency components."""
    image_np = np.array(image)
    shifted = np.fft.fftshift(np.fft.fft2(image_np))
    rows, cols = shifted.shape[:2]
    row_shift, col_shift = np.random.randint(-max_shift, max_shift + 1, size=2)
    
    rolled = np.roll(np.roll(shifted, row_shift, axis=0), col_shift, axis=1)
    inverse_fft = np.real(np.fft.ifft2(np.fft.ifftshift(rolled)))
    
    shifted_image = np.clip(inverse_fft, 0, 255).astype(np.uint8)
    return TF.to_pil_image(shifted_image)

aug_list = [
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.Lambda(lambda img: add_gaussian_noise(img, noise_factor=0.05)),
    transforms.Lambda(lambda img: add_frequency_shift(img, max_shift=5))
]

def mixup(image1, image2, alpha=0.2):
    """Blend two DFT images, keeping their frequency characteristics."""
    lambda_param = np.random.beta(alpha, alpha)
    img1_array = np.array(image1, dtype=np.float32)
    img2_array = np.array(image2.resize(image1.size), dtype=np.float32)
    
    mixed_img_array = lambda_param * img1_array + (1 - lambda_param) * img2_array
    return Image.fromarray(np.uint8(np.clip(mixed_img_array, 0, 255)))

def augmix(image, width=3, depth=-1, alpha=1.0):
    """Apply a chain of random augmentations, focusing on DFT perturbations."""
    ws = np.random.dirichlet([alpha] * width).astype(np.float32)
    m = np.random.beta(alpha, alpha)

    mix = np.zeros_like(np.array(image), dtype=np.float32)
    for i in range(width):
        image_aug = image.copy()
        d = depth if depth > 0 else np.random.randint(1, 4)
        for _ in range(d):
            op = np.random.choice(aug_list)
            image_aug = op(image_aug)
        mix += ws[i] * np.array(image_aug, dtype=np.float32)

    mixed = (1 - m) * np.array(image, dtype=np.float32) + m * mix
    return Image.fromarray(np.uint8(np.clip(mixed, 0, 255)))

def augment_images(input_dir, output_dir, num_augments=5):
    image_filenames = [f for f in os.listdir(input_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for img_file in image_filenames:
        img_path = os.path.join(input_dir, img_file)
        img = Image.open(img_path).convert('L')  # Grayscale for DFT

        for i in range(num_augments):
            aug_img = augmix(img)

            if len(image_filenames) > 1:
                other_img_file = random.choice(image_filenames)
                other_img_path = os.path.join(input_dir, other_img_file)
                other_img = Image.open(other_img_path).convert('L')

                mixup_img = mixup(aug_img, other_img)

                output_img_path = os.path.join(output_dir, f'{os.path.splitext(img_file)[0]}_aug_{i+1}.png')
                mixup_img.save(output_img_path, format='PNG')

augment_images(input_dir, output_dir, num_augments=5)